In [ ]:
import gymnasium as gym
import ale_py
gym.register_envs(ale_py)

from stable_baselines3 import DQN
from stable_baselines3.common.atari_wrappers import AtariWrapper
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.vec_env import VecFrameStack, DummyVecEnv, VecTransposeImage
from stable_baselines3.common.monitor import Monitor

# Crear el ambiente de Breakout y su pre-procesamiento
def make_env():
    env = gym.make("ALE/Breakout-v5", render_mode=None)
    env = AtariWrapper(env)
    env = Monitor(env)  # Monitorear episodios recompensas, etc.
    return env

# Vectorizar el ambiente
env = DummyVecEnv([make_env])
env = VecFrameStack(env, n_stack=4)  # Apilar 4 frames
env = VecTransposeImage(env)  # Primero los canales para PyTorch

# Inicializar el modelo
model = DQN("CnnPolicy", env, verbose=0, buffer_size=300_000, learning_starts=30_000, batch_size = 32,
            learning_rate=1e-4, train_freq=4, target_update_interval=3_000, exploration_fraction=0.1,
            exploration_final_eps = 0.01, gamma=0.99, tensorboard_log = "./dqn_breakout_tensorboard/")

# Train the model
model.learn(total_timesteps=300_000, progress_bar = True)

# Evaluate the model
mean_reward, std_reward = evaluate_policy(model, env, n_eval_episodes=10, render=False)
print(f"Recompensa promedio: {mean_reward} ± {std_reward}")

# Guardar el modelo
model.save("dqn_breakout")


C:\Users\Josue Santana Robled\.conda\envs\actumlogos_drl\lib\site-packages\stable_baselines3\common\buffers.py:242: UserWarning: This system does not have apparently enough memory to store the complete replay buffer 16.94GB > 8.18GB
  warnings.warn(


C:\Users\Josue Santana Robled\.conda\envs\actumlogos_drl\lib\site-packages\rich\live.py:260: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

Output()

In [7]:
%load_ext tensorboard
%tensorboard --logdir ./dqn_breakout_tensorboard/

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


Reusing TensorBoard on port 6006 (pid 21528), started 1:41:15 ago. (Use '!kill 21528' to kill it.)

In [6]:
import imageio
from PIL import Image

# Crear el ambiente de Breakout y su pre-procesamiento
def make_env():
    env = gym.make("ALE/Breakout-v5", render_mode='rgb_array')
    env = AtariWrapper(env)
    env = Monitor(env)  # Monitorear episodios recompensas, etc.
    return env

# Vectorizar el ambiente
env = DummyVecEnv([make_env])
env = VecFrameStack(env, n_stack=4)  # Apilar 4 frames
env = VecTransposeImage(env)  # Primero los canales para PyTorch

model = DQN.load("dqn_breakout")

# Desired size (width, height)
resize_to = (1000, 1000)

obs = env.reset()
done = False
frames = []

while not done:
    frame = env.render()
    img = Image.fromarray(frame)
    resized_frame = img.resize(resize_to, Image.Resampling.LANCZOS)
    frames.append(resized_frame)
    action, _ = model.predict(obs)
    obs, _, done, _ = env.step(action)

# Save video
imageio.mimsave("breakout_agent.gif", frames, fps=30)

C:\Users\e807575\AppData\Local\anaconda3\envs\atari\lib\site-packages\stable_baselines3\common\buffers.py:242: UserWarning: This system does not have apparently enough memory to store the complete replay buffer 16.94GB > 3.63GB
  warnings.warn(
